# Results  

Final results from the pipeline are stored in subdirectory *plateanalysis/footprints/results/*. 

Each HTML file contains the result of a pipeline run over a *sequence* of plates. Sequences of plates are defined in file *settings.py*, and created based on the output of notebook *footprints.ipynb*.

This notebook creates file *candidates_all.fits*, which contains a table with all candidates found at all plate pairs and sequences. It contains the summarized result of the entire project to date.

HTML files contain just the very final product for each sequence; intermediate results are not stored in GitHub due to their policy of minimizing the storage of files that are either temporary, or potentially large, such as FITS binary tables and such. Intermediate results can be seen only by running the code locally over the appropriate data sets.  

#### Note

Due to sheer size, only one of the plate sequences I worked with has its pipeline output uploaded to the repo. Also, GitHub is unable to display the html due to its size. Download the file and use your browser locally to display the file.  

Alternatively, you can download the result files from Google Drive: https://drive.google.com/drive/folders/164xc4faMDbPt84ZileJZoHzK94kvG8Gm


Below is a summary of findings to date. Numbers refer to the plate ID code in the APPLAUSE archive.

- Data from the **1m-Spiegelteleskop** and the **Doppel-Reflektor** telescopes: still to be reviewed in light of the upgraded software.

- Data from the **Grosser Schmidt-Spiegel** telescope yeld a few candidates.




### Notes

APPLAUSE scans were performed on the original plates, not copies.

## Processing stats

Here, we build a summary of number of detections/objects for each plate pair, plus plate metadata information. The goal is to provide a bird's view of the coverage of the project to date, as well as a complete list of candidates in a single table.

In [1]:
import os
import warnings

from astropy.table import Table, vstack, hstack
from astropy.time import Time
from astropy.utils.exceptions import AstropyWarning

import pandas as pd
from erfa import ErfaWarning

from library import get_earth_shadow
import settings
from settings import CATALOG, RESULTS, get_parameters, fname, sequences

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
warnings.simplefilter('ignore', category=AstropyWarning)
warnings.simplefilter('ignore', category=ErfaWarning)
warnings.simplefilter('ignore', category=FutureWarning)

In [3]:
table_header = {
    'summary': {
        'names': ['seq', 'plate 1', 'ut_mid_1', 'exptime_1', 'es_1','plate 2', 'ut_mid_2', 
                  'exptime_2', 'es_2', 'total sources', 'matched', 'non-matched', 'candidates'],
        'dtypes': ['S1', 'S1', 'S1', 'i4', 'f4', 'S1', 'S1', 'i4', 'f4', 'i4', 'i4', 'i4', 'i4'],
    },
    'metadata': {
        'names': ['ut_mid', 'exptime', 'es', 'plate_id_next', 'ut_mid_next', 'exptime_next', 'es_next'],
        'dtypes': ['S1', 'i4', 'f4', 'i4', 'S1', 'i4', 'f4'],
    },
}

In [4]:
def make_table(table_type):
    table = Table(names=table_header[table_type]['names'],
                  dtype=table_header[table_type]['dtypes'])
    return table

In [5]:
# use Pandas to display table so all rows can be displayed at once. Displaying long
# Astropy tables directly causes only the first and last few rows to be displayed. 

def display_with_pandas(table):
    df = table.to_pandas()

    # avoid display of unicode reminder in every string field
    df['seq'] = df['seq'].apply(lambda x: x.decode('utf-8'))
    df['plate 1'] = df['plate 1'].apply(lambda x: x.decode('utf-8'))
    df['plate 2'] = df['plate 2'].apply(lambda x: x.decode('utf-8'))
    df['ut_mid_1'] = df['ut_mid_1'].apply(lambda x: x.decode('utf-8'))
    df['ut_mid_2'] = df['ut_mid_2'].apply(lambda x: x.decode('utf-8'))

    # style with CSS properties for table headers (th) and data cells (td)
    th_props = [('font-size', '12px'), ('text-align', 'right'), ('font-weight', 'bold')]
    td_props = [('font-size', '11px'), ('text-align', 'right')]
    styles = [dict(selector="th", props=th_props),dict(selector="td", props=td_props)]
    styled_df = df.style.set_table_styles(styles)    
    
    format_mapping = {
        'es_1': '{:.3}'.format,
        'es_2': '{:.3}'.format,
    }
    styled_df = styled_df.format(format_mapping)    

    display(styled_df)

In [6]:
# get metadata for a given plate. Metadata is taken from original input table from applause.

def get_metadata(plate, catalog_table):
    
    mask = catalog_table['plate_id'] == plate
    t = catalog_table[mask]
    
    long = t['site_longitude'][0]
    lat  = t['site_latitude'][0]
    ra  = t['ra_icrs'][0]
    dec = t['dec_icrs'][0]
    time_event = Time(t['ut_mid'][0])

    es, _ = get_earth_shadow(ra, dec, time_event, longitude=long, latitude=lat)
    
    return t['ut_mid'][0], t['exptime'][0], es

In [7]:
def add_metadata(metadata, table):
    
    # create temporary table to contain metadata
    meta_table = make_table('metadata')
    
    for row_index in range(len(table)):
        plate_id = table['plate_id_1'][row_index]

        plate_id_next = metadata[plate_id]['plate_id_next']    
        ut_mid        = metadata[plate_id]['ut_mid']    
        exptime       = metadata[plate_id]['exptime']    
        es            = metadata[plate_id]['es']    
        ut_mid_next   = metadata[plate_id]['ut_mid_next']    
        exptime_next  = metadata[plate_id]['exptime_next']    
        es_next       = metadata[plate_id]['es_next']    
    
        meta_table.add_row([ut_mid, exptime,es, plate_id_next, ut_mid_next, exptime_next, es_next])
  
    # join with main table
    result = hstack([meta_table, table])
    
    return result    

In [8]:
# metadata comes from original input table from applause
catalog_table = Table.read(CATALOG, format='ascii.csv')

# summary table
result = make_table('summary')

# list of tables to collect all candidate events
candidates_list = []

# collect some metadata here
metadata = {}

In [9]:
for seq_key in list(sequences.keys()):
    seq = sequences[seq_key]

    for i in range(len((seq)) - 1):
        try:
            plate_id = seq[i]
            next_plate_id = seq[i+1]
            
            # metadata
            ut_mid_1, exptime_1, es_1 = get_metadata(plate_id, catalog_table)
            ut_mid_2, exptime_2, es_2 = get_metadata(next_plate_id, catalog_table)
            
            metadata[plate_id] = {
                'ut_mid': ut_mid_1,
                'exptime': exptime_1,
                'es': es_1,
                'plate_id_next': next_plate_id,
                'ut_mid_next': ut_mid_2,
                'exptime_next': exptime_2,
                'es_next': es_2,
            }
            
            plate_id_str = str(plate_id)
            next_plate_id_str = str(next_plate_id)

            key = plate_id_str + ',' + next_plate_id_str
            par = get_parameters(key)

            # row count on each relevant product table
            table_sources = Table.read(fname(par['table1']), format='ascii.csv')
            mask = table_sources['scan_id'] == table_sources['scan_id'][0]
            table_sources = table_sources[mask]
            sources = len(table_sources)

            table_matched = Table.read(fname(par['table_matched']), format='fits')
            matched = len(table_matched)

            table_non_matched = Table.read(fname(par['table_non_matched']), format='fits')
            non_matched = len(table_non_matched)

            table_candidates = Table.read(fname(par['table_candidates']), format='fits')
            candidates = len(table_candidates)
            
            # add this candidates table to final product
            candidates_list.append(table_candidates)
            
            result.add_row([seq_key, plate_id_str, ut_mid_1, exptime_1, es_1,
                            next_plate_id_str, ut_mid_2, exptime_2, es_2,
                            sources, matched, non_matched, candidates])
        except KeyError:
            break

### Summary table

Table columns:

- **seq**: sequence ID generated by notebook *footprints.ipynb*
- **plate ...**: plate IDs defining the pair of plates
- **ut_mid ...**: UT mid for each plate
- **exptime ...**: EXPTIME for each plate
- **es ...**: Earth's shadow angle (deg) for each plate
- **total sources**: total number of sources in the APPLAUSE table of the first plate, for the *_x* scan direction
- **matched**: number of sources in the first plate that have a match in the second plate, after the first plate's source table gets prunned of bad detections (as per sextractor-generated parameters)
- **non-matched**: number of sources in the first plate (after it was cleaned up as above) that have **NO** match on the second plate
- **candidates**: number of candidates remaining after filtering is applyied (filtering based on Gaussian fitting, radial profile, circularity, and aperture photometry analysis)


In [10]:
display_with_pandas(result)

,seq,plate 1,ut_mid_1,exptime_1,es_1,plate 2,ut_mid_2,exptime_2,es_2,total sources,matched,non-matched,candidates
0,seq01,9245,1956-09-25 19:00:47,1800,69.3,9246,1956-09-25 19:43:40,1800,68.4,590001,246737,27726,1
1,seq01,9246,1956-09-25 19:43:40,1800,68.4,9247,1956-09-25 20:29:33,1800,67.6,465264,233033,16063,16
2,seq03,9313,1956-12-03 17:15:31,900,62.9,9315,1956-12-03 18:06:23,900,61.8,55239,17234,93,0
3,seq03,9315,1956-12-03 18:06:23,900,61.8,9317,1956-12-03 18:56:48,900,60.6,76057,22462,2029,1
4,seq03,9317,1956-12-03 18:56:48,900,60.6,9318,1956-12-03 19:26:11,900,60.0,59981,21413,426,0
5,seq03,9318,1956-12-03 19:26:11,900,60.0,9319,1956-12-03 20:27:18,900,58.8,62314,21293,139,1
6,seq03,9319,1956-12-03 20:27:18,900,58.8,9320,1956-12-03 20:55:56,900,58.3,75926,23992,380,1
7,seq04,9322,1956-12-06 18:54:56,900,61.9,9323,1956-12-06 19:19:52,900,61.4,58894,20801,315,2
8,seq04,9323,1956-12-06 19:19:52,900,61.4,9324,1956-12-06 19:44:24,900,60.9,55417,21108,475,0
9,seq04,9324,1956-12-06 19:44:24,900,60.9,9325,1956-12-06 20:08:14,900,60.4,54232,18804,97,0


## Final candidates table

All candidates from all plate pairs are stored together into a single FITS table. Add some metadata to help navigating the table.

In [11]:
candidates_table = vstack(candidates_list)

candidates_table = add_metadata(metadata, candidates_table)

filename = os.path.join(RESULTS, "candidates_all.fits")

candidates_table.write(filename, overwrite=True)

In [12]:
candidates_table

ut_mid,exptime,es,plate_id_next,ut_mid_next,exptime_next,es_next,source_id,process_id_1,scan_id_1,plate_id_1,archive_id_1,solution_num,annular_bin_1,dist_center_1,dist_edge_1,sextractor_flags_1,model_prediction_1,ra_icrs,dec_icrs,ra_error,dec_error,gal_lon,gal_lat,ecl_lon,ecl_lat,x_sphere,y_sphere,z_sphere,healpix256,healpix1024,nn_dist,zenith_angle,airmass,natmag,natmag_error,bpmag,bpmag_error,rpmag,rpmag_error,natmag_plate,natmag_correction,natmag_residual,phot_range_flags,phot_calib_flags,color_term,cat_natmag,match_radius,gaiaedr3_id,gaiaedr3_gmag,gaiaedr3_bp_rp,gaiaedr3_dist,gaiaedr3_neighbors,timestamp_insert_1,timestamp_update_1,pos,process_id_2,scan_id_2,plate_id_2,archive_id_2,source_num,x_source,y_source,a_source,b_source,theta_source,erra_source,errb_source,errtheta_source,elongation,x_peak,y_peak,flag_usepsf,x_image,y_image,erra_image,errb_image,errtheta_image,x_psf,y_psf,erra_psf,errb_psf,errtheta_psf,mag_auto,magerr_auto,flux_auto,fluxerr_auto,mag_iso,magerr_iso,flux_iso,fluxerr_iso,flux_max,flux_radius,isoarea,sqrt_isoarea,background,sextractor_flags_2,dist_center_2,dist_edge_2,annular_bin_2,flag_rim,flag_negradius,flag_clean,model_prediction_2,timestamp_insert_2,timestamp_update_2,next_plate_id,id,group_id,group_size,local_bkg,x_init,y_init,flux_init,fwhm_init,x_fit,y_fit,flux_fit,fwhm_fit,x_err,y_err,flux_err,fwhm_err,npixfit,qfit,cfit,flags,profile_diff,circularity,area,shape_defect,circle_deviation
bytes19,int32,float32,int32,bytes19,int32,float32,int64,int64,int64,int64,int64,int64,int64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,float64,float64,float64,int64,int64,float64,float64,int64,bytes29,bytes29,bytes42,int64,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,int64,float64,float64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,int64,float64,float64,int64,int64,int64,int64,float64,bytes29,bytes29,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,int64,float64,float64,float64,float64,float64
1956-09-25 19:00:47,1800,69.34504,9246,1956-09-25 19:43:40,1800,68.43947,40348540307131,34854,12125,9245,102,0,3,6319.804,5091.97,0,1.0,300.34019624173004,36.06297189920122,0.19533699750900269,0.15063799917697906,72.38946406162688,2.998909759709035,315.161172856056,54.848941568802424,0.4083347893100495,-0.697657186432828,0.5886740609690116,234141,3746265,14.65872859954834,17.533716201782227,1.0485596656799316,--,--,--,--,--,--,0.0,--,--,0,0,--,--,0.0,--,--,--,--,0,2022-06-13 20:51:34.067699+00,2022-06-13 20:51:34.067699+00,"(5.24192530050409 , 0.629417597695254)",34854,12125,9245,102,307131,5091.97,12546.520350217868,1.2191925048828125,1.1855409145355225,82.49700164794922,0.04974868893623352,0.049733035266399384,-20.831024169921875,1.028385043144226,5092,12548,0,5091.97,12548.012,0.04974868893623352,0.049733035266399384,-20.831024169921875,--,--,--,--,--,11.53481674194336,0.03957675024867058,243261.484375,8865.1005859375,11.936367988586426,0.03361285850405693,168055.515625,5201.49755859375,12783.9482421875,2.058568239212036,21,4.582575798034668,31350.048828125,0,6319.804,5091.97,3,0,0,1,1.0,2022-06-13 05:02:56.944805+00,2022-06-13 05:02:56.944805+00,9246,7877,7877,1,0.0,5091.97,12546.520350217868,756784.7543689883,8.0,5090.912310595922,12547.05741644342,292134.557300108,4.447060067124626,0.09575663530838362,0.10517358167429539,14353.180360962919,0.16495802167195556,961,2.963307066098008,-0.0011132072357823498,0,0.03655125072418025,0.8826102797989687,24.0,0.06361607142857142,0.08841330594894234
1956-09-25 19:43:40,1800,68.43947,9247,1956-09-25 20:29:33,1800,67.580

## To do

- visual vetting
- augment dataset with: (i) sequences with two plates only; (ii) sequences that reach over multiple nights.
- look for alignments.

- fix *footprints.ipynb*: time order in sequences of plates. This is relevant only for long sequences, so low priority for now.

- multivariate probability - NO NEED for now.
- add circularity diagnostic - DONE
- add other shape diagnostics - DONE
- profile metrics - DONE
- refactor towards batch processing - DONE
- automate data download - DONE
- add Earth shadow angle to tables - DONE

## Summary of possible bright event in plate 9319

### Needs revision: other similar events exist - do it after visual vetting

- date 1956-12-03
- time between utc_mid: 28 min
- exposure time on each plate: 15 min
- emulsion Kodak Oa-E both plates

Plates 9313, 9315, 9316, 9317, 9318, were searched for similar events, with no detection. These plates were all taken on the same night of 1956-12-03 over a 4 hr time span, with the same telescope pointing (except 9316 that was taken 24 hr later). 

The event position on the plate (distance to center = 3248.3 px, distance to edge = 1061.8 px, annular bin = 4) places it in a region where artifacts can be perhaps more likely to happen. The profile suggests that it might be indeed an artifact: these tend to have systematically narrower widths than stars of same brightness. Or, could be indeed the result of a very brief flash of light: these are argued to appear with somewaht narrower PSFs than long-exposure star images.

Plate's FOV is far from Earth's shadow. 